# ST-OMR Meter V5-1 — 30 TRAIN BBox Pilot

**Güvenlik sınırı:** Yalnız `TEST/METER_V2_1500_PACKAGE_AB_CLEAN` kullanılır. Precheck canlı heartbeat ile arka plan worker içinde çalışır. Final holdout kilitli; training/tuning/model/inference kapalıdır.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, pathlib, subprocess, sys, shutil
EXPECTED_CODE_SHA = '0146ecae4f3abb06175872bebc6b7f15644b4773'
BRANCH = 'fix/meter-v5-1-clean-bbox-pilot'
REPO_URL = 'https://github.com/khfy7wpr5p-maker/st-omr-training.git'
REPO_DIR = pathlib.Path('/content/st-omr-training-v5-1')
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
subprocess.run(['git','clone','--depth','20','--branch',BRANCH,REPO_URL,str(REPO_DIR)], check=True)
subprocess.run(['git','-C',str(REPO_DIR),'checkout','--detach',EXPECTED_CODE_SHA], check=True)
actual = subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','HEAD'], text=True).strip()
assert actual == EXPECTED_CODE_SHA, (actual, EXPECTED_CODE_SHA)
subprocess.run([sys.executable,'-m','pip','install','-q','Pillow==12.3.0'], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path: sys.path.insert(0, str(REPO_DIR))
print('CODE_PIN=PASS', actual)


## Precheck — TEST seçimi + canlı izleme + final_holdout lock

Kullanıcı TEST altındaki kopyayı otorite olarak seçti. Bu hücre tüm MyDrive'ı yeniden taramaz. Exact seçilmiş kökü doğrular; dataset gate'i arka plan thread'inde çalıştırır ve her 10 saniyede heartbeat basar. Worker bitmeden annotation açılmaz.


In [ ]:
import pathlib, threading, time
from st_omr_training.meter_v5_1_bbox_pilot import verify_dataset_structure, ensure_final_holdout_lock, DATASET_NAME

MYDRIVE = pathlib.Path('/content/drive/MyDrive')
DATA_ROOT = MYDRIVE / 'TEST' / DATASET_NAME
print('===== METER V5-1 LIVE PRECHECK =====')
print('[1/4] selected authority:', DATA_ROOT)
if not MYDRIVE.is_dir(): raise RuntimeError(f'MyDrive not mounted: {MYDRIVE}')
if not DATA_ROOT.is_dir(): raise RuntimeError(f'Selected TEST dataset not found: {DATA_ROOT}')
print('[1/4] TEST dataset selection ........ PASS')

_state = {'phase':'starting','result':None,'error':None}
def _precheck_worker():
    try:
        _state['phase'] = 'verify_dataset_structure'
        gate = verify_dataset_structure(DATA_ROOT)
        _state['phase'] = 'ensure_final_holdout_lock'
        lock_path = ensure_final_holdout_lock(DATA_ROOT, gate)
        _state['result'] = (gate, lock_path)
        _state['phase'] = 'done'
    except BaseException as exc:
        _state['error'] = exc
        _state['phase'] = 'error'

_thread = threading.Thread(target=_precheck_worker, name='meter-v5-1-precheck', daemon=True)
_started = time.monotonic()
_thread.start()
while _thread.is_alive():
    elapsed = int(time.monotonic() - _started)
    print(f'[LIVE] phase={_state["phase"]} elapsed={elapsed}s | FINAL_HOLDOUT=LOCKED | TRAINING=CLOSED | MODEL=CLOSED', flush=True)
    _thread.join(timeout=10)
_thread.join()
if _state['error'] is not None:
    print('[PRECHECK] FAIL-CLOSED:', repr(_state['error']))
    raise _state['error']
GATE, LOCK_PATH = _state['result']
print('[2/4] dataset structure/count gate ... PASS')
print('[3/4] leakage/provenance gate ........ PASS')
print('[4/4] final_holdout lock ............. PASS')
print('DATASET_GATE=PASS')
print('data_root=', DATA_ROOT)
print('total=', GATE['total'])
print('unique_family_id=', GATE['unique_family_id'])
print('unique_sample_id=', GATE['unique_sample_id'])
print('unique_source_image=', GATE['unique_source_image'])
print('package_ab_only=', GATE['package_ab_only'])
print('cross_split_family_leakage=', GATE['cross_split_family_leakage'])
print('cross_meter_family_overlap=', GATE['cross_meter_family_overlap'])
for split in ('train','val','final_holdout'):
    print(split.upper(), '2/4=', GATE['directory_counts'][f'{split}/2_4'], '3/4=', GATE['directory_counts'][f'{split}/3_4'], '4/4=', GATE['directory_counts'][f'{split}/4_4'])
print('final_holdout_locked=', GATE['final_holdout_locked'])
print('final_holdout_lock=', LOCK_PATH)
print('ANNOTATION_SCOPE=train_pilot_30_only')
print('MODEL_OPENED=False; TRAINING=False; TUNING=False; INFERENCE_COUNT=0')


## 30 TRAIN interaktif BBox pilotu

Yalnız `DATASET_GATE=PASS` sonrası çalıştır. Tek BBox: üst+alt meter rakamları tamamen içeride; clef/key signature/ilk nota mümkün olduğunca dışarıda. Her kayıt Drive'a checkpoint edilir; resume `sample_id` ile yapılır.


In [ ]:
from st_omr_training.meter_v5_1_bbox_pilot_colab import launch_colab_pilot
assert GATE['total'] == 1500 and GATE['final_holdout_locked'] is True
SESSION = launch_colab_pilot(data_root=str(DATA_ROOT))
print('PILOT_UI=READY')
print('resume_index=', SESSION.resume_index())
print('handled=', SESSION.handled_count, 'pass=', SESSION.pass_count, 'review=', SESSION.review_count)
print('FINAL_HOLDOUT=LOCKED; TRAINING=False; MODEL_OPENED=False; INFERENCE_COUNT=0')


## Pilot audit — yalnız 30/30 sonrası

Audit insan kutularını değiştirmez; mekanik kontrolleri ve boyut istatistiklerini raporlar.


In [ ]:
import json
from st_omr_training.meter_v5_1_bbox_pilot import write_pilot_audit
AUDIT_PATH = write_pilot_audit(DATA_ROOT)
AUDIT = json.loads(AUDIT_PATH.read_text(encoding='utf-8'))
print(json.dumps(AUDIT, indent=2, sort_keys=True))
print('AUDIT_PATH=', AUDIT_PATH)
print('FINAL_HOLDOUT=LOCKED; TRAINING=False; MODEL_OPENED=False; INFERENCE_COUNT=0')
